### Load Data

In [1]:
from sklearn import datasets
housing = datasets.fetch_california_housing()

print(housing.feature_names)

x = housing.data # input features - independent variables
y = housing.target # price

print([x[0]]) # print the first row of the input features
print([y[0]]) # print the first value of the target variable

['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
[array([   8.3252    ,   41.        ,    6.98412698,    1.02380952,
        322.        ,    2.55555556,   37.88      , -122.23      ])]
[4.526]


### Split Data

In [2]:
# 20/80

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
                                            x, 
                                            y, 
                                            test_size=0.2, 
                                            random_state=42
                                            )

### Algorithm

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

model = LinearRegression()
model.fit(X_train, y_train) # Train the model on the training data

# how do we test it?
y_pred = model.predict(X_test) # Predict on the test data - remaining 20%

r2 = r2_score(y_test, y_pred) # R² score - how well the model explains the variance in the data
mse = mean_squared_error(y_test, y_pred) # MSE - average squared difference
print(f"R² Score: {r2:.4f}")
print(f"Mean Squared Error: {mse:.4f}")

R² Score: 0.5758
Mean Squared Error: 0.5559


### Polynomial features

In [4]:
# used to expand the feature space by adding polynomial features, 
# allowing linear models to capture non-linear relationships.

from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures() # default is degree=2, which adds squared terms and interaction terms

x = poly.fit_transform(x)

X_train, X_test, y_train, y_test = train_test_split(
                                            x, 
                                            y, 
                                            test_size=0.2, 
                                            random_state=42
                                            )

model.fit(X_train, y_train) # Train the model on the training data

# how do we test it?
y_pred = model.predict(X_test) # Predict on the test data - remaining 20%

r2 = r2_score(y_test, y_pred) # R² score - how well the model explains the variance in the data
mse = mean_squared_error(y_test, y_pred) # MSE - average squared difference
print(f"R² Score: {r2:.4f}")
print(f"Mean Squared Error: {mse:.4f}")

R² Score: 0.6457
Mean Squared Error: 0.4643


### Logistic Regression, GBR and RBR

In [5]:
from sklearn.ensemble import (HistGradientBoostingRegressor, RandomForestRegressor)

LR = LinearRegression()
# Create an instance of the Linear Regression model.
# This acts as a simple baseline algorithm.

GBR = HistGradientBoostingRegressor()
# DIFFERENCE from previous step:
# Instead of GradientBoostingRegressor, we now use HistGradientBoostingRegressor.
# This algorithm is optimized for speed and large datasets.

RBR = RandomForestRegressor()
# Create an instance of the Random Forest Regressor.
# This algorithm builds multiple decision trees and averages their predictions.

n_jobs = -1
# NEW compared to the previous version:
# n_jobs controls how many CPU cores are used for computation.
# -1 means use ALL available CPU cores to speed up training.

for i in [LR, GBR, RBR]:
# Loop through each model to train and evaluate them.
# This allows comparison of multiple algorithms using the same dataset.

    i.fit(X_train, y_train)
    # Train the model using the training dataset.
    # The algorithm learns patterns between input features and house prices.

    # How do we know the model has learned?
    y_pred = i.predict(X_test)
    # Use the trained model to make predictions on the unseen test dataset.

    r2 = r2_score(y_test, y_pred)
    # Calculate the R² score to measure model performance.

    print(i, r2)
    # Print the model name and its R² score.
    # This helps compare which algorithm performs best on the dataset.

LinearRegression() 0.6456819832921057
HistGradientBoostingRegressor() 0.8308654063002674
RandomForestRegressor() 0.7975097335673524


### Fine tunning HistGradientBoostingRegressor

In [7]:
for i in [100,150,200,250,300,350]:
# DIFFERENCE from previous step:
# Instead of comparing multiple algorithms, we now tune a parameter of the best model.
# Here we test different values of "max_iter".
# max_iter represents the number of boosting iterations (similar to number of trees).

    model = HistGradientBoostingRegressor(max_iter = i)
    # Create the model with a specific value of max_iter.
    # Each iteration adds another tree to improve the model.

    model.fit(X_train, y_train)
    # Train the model using the training dataset.

    # How do we know the model has learned?
    y_pred = model.predict(X_test)
    # Use the trained model to predict house prices on the test dataset.

    r2 = r2_score(y_test, y_pred)
    # Calculate the R² score to measure model performance.

    print(i, r2)
    # Print the number of iterations and the corresponding R² score.
    # This helps identify which parameter value gives the best performance.

100 0.8311063744384598
150 0.8373457684071572
200 0.8407240277199228
250 0.8416374888979583
300 0.8424402485868265
350 0.8420476354438917


### Fine tuning Hyperparameters

In [8]:
# Small adjustments within the algorithm that can significantly impact performance.
# Finding relation between X and Y

for j in [0.1, 0.05, 0.001]:
# DIFFERENCE from previous step:
# We now tune the learning_rate parameter.
# learning_rate controls how much each tree contributes to the model.

    for i in [200, 250, 300]:
    # Loop through different numbers of trees (max_iter).
    # In the previous step we observed performance stabilised around 200.

        model = HistGradientBoostingRegressor(max_iter=i, learning_rate=j)
        # Create the model using two tuned parameters:
        # max_iter → number of boosting trees
        # learning_rate → how quickly the model learns

        model.fit(X_train, y_train)
        # Train the model using the training dataset.

        # How do we know the model has learned?
        y_pred = model.predict(X_test)
        # Use the trained model to make predictions on the test dataset.

        r2 = r2_score(y_test, y_pred)
        # Calculate the R² score to measure model performance.

        print(i, j, r2)
        # Print the number of trees, learning rate, and corresponding R² score.
        # This helps identify the best parameter combination.

200 0.1 0.8411918740296238
250 0.1 0.8435652562842988
300 0.1 0.8425666656947494
200 0.05 0.8322100473324043
250 0.05 0.8369021054423169
300 0.05 0.8398456062223036
200 0.001 0.21552631523210763
250 0.001 0.25686537068744375
300 0.001 0.29715696106533107
